In [0]:
%sql
-- Check current DBR capabilities
SELECT 
  'DBR Version' as info_type,
  current_version().dbr_version as value
UNION ALL
SELECT 
  'Delta Lake Features',
  CASE WHEN current_version().dbr_version LIKE '%delta%' THEN 'Available' ELSE 'Limited' END
UNION ALL
SELECT
  'Photon Status',
  CASE WHEN current_version().dbr_version LIKE '%photon%' THEN 'Enabled' ELSE 'Disabled' END;

info_type,value
DBR Version,17.1.x-aarch64-photon-scala2.13
Delta Lake Features,Limited
Photon Status,Enabled


In [0]:
%sql
USE CATALOG demo_catalog

In [0]:
%sql
USE SCHEMA raw

In [0]:
%sql
-- Test advanced SQL features available in newer DBR (ONLY Run at the start of once need shcema changes)
CREATE OR REPLACE TABLE dbr_feature_test (
  id BIGINT,
  category STRING,
  value DECIMAL(10,2),
  created_date DATE
) USING DELTA;

In [0]:
%sql
-- Insert test data with DBR optimizations (ONLY Execute if you need new data generation)
INSERT INTO dbr_feature_test
SELECT 
  id,
  CASE WHEN id % 5 = 0 THEN 'Premium'
       WHEN id % 3 = 0 THEN 'Standard' 
       ELSE 'Basic' END as category,
  CAST(RANDOM() * 1000 AS DECIMAL(10,2)) as value,
  DATE_ADD(CURRENT_DATE(), CAST(-(id % 365) AS INT)) as created_date
FROM RANGE(100000);


num_affected_rows,num_inserted_rows
100000,100000


In [0]:

%sql

-- Test SQL features that benefit from newer DBR
-- Window functions with optimization
SELECT 
  category,
  created_date,
  value,
  AVG(value) OVER (
    PARTITION BY category 
    ORDER BY created_date 
    ROWS BETWEEN 7 PRECEDING AND CURRENT ROW
  ) as moving_avg_7day,
  RANK() OVER (
    PARTITION BY category 
    ORDER BY value DESC
  ) as value_rank
FROM dbr_feature_test
WHERE created_date >= CURRENT_DATE() - INTERVAL 30 DAYS
ORDER BY category, created_date;


category,created_date,value,moving_avg_7day,value_rank
Basic,2025-09-02,997.89,997.890000,10
Basic,2025-09-02,996.54,997.215000,21
Basic,2025-09-02,989.96,994.796667,45
Basic,2025-09-02,985.46,992.462500,60
Basic,2025-09-02,983.54,990.678000,74
Basic,2025-09-02,977.74,988.521667,97
Basic,2025-09-02,975.72,986.692857,107
Basic,2025-09-02,972.20,984.881250,127
Basic,2025-09-02,960.35,980.188750,166
Basic,2025-09-02,960.05,975.627500,169


In [0]:
%sql
-- Test adaptive query execution benefits
SELECT 
  t1.category,
  COUNT(*) as record_count,
  AVG(t1.value) as avg_value,
  SUM(t2.value) as related_sum
FROM dbr_feature_test t1
JOIN (
  SELECT category, value 
  FROM dbr_feature_test 
  WHERE value > 500
) t2 ON t1.category = t2.category
GROUP BY t1.category
HAVING COUNT(*) > 1000
ORDER BY avg_value DESC;

category,record_count,avg_value,related_sum
Standard,355631112,500.336825,266390026633.71
Basic,1423884434,500.111579,1068304146524.02
Premium,199680000,499.698226,149346446800.00


In [0]:
%sql
SELECT * FROM dbr_feature_test 
WHERE category = 'Premium' 
AND created_date >= CURRENT_DATE() - INTERVAL 30 DAYS;

id,category,value,created_date
25185,Premium,291.97,2025-10-01
25190,Premium,300.93,2025-09-26
25195,Premium,978.76,2025-09-21
25200,Premium,921.31,2025-09-16
25205,Premium,857.84,2025-09-11
25210,Premium,570.05,2025-09-06
25215,Premium,269.23,2025-09-01
25550,Premium,8.83,2025-10-01
25555,Premium,679.77,2025-09-26
25560,Premium,18.90,2025-09-21


In [0]:
%sql
-- Test Delta Lake time travel (DBR enhancement)
-- Update records to create new version
UPDATE dbr_feature_test 
SET value = value * 1.1 
WHERE category = 'Premium' AND created_date >= CURRENT_DATE() - INTERVAL 30 DAYS;

num_affected_rows
1918


In [0]:
%sql
-- Query historical data
SELECT 
  'Current' as version_type,
  category,
  COUNT(*) as record_count,
  AVG(value) as avg_value
FROM dbr_feature_test
WHERE category = 'Premium'
GROUP BY category

UNION ALL

SELECT 
  'Version 0' as version_type,
  category,
  COUNT(*) as record_count, 
  AVG(value) as avg_value
FROM dbr_feature_test VERSION AS OF 1
WHERE category = 'Premium' 
GROUP BY category;

version_type,category,record_count,avg_value
Current,Premium,20000,504.493182
Version 0,Premium,20000,499.698226


In [0]:
%sql
-- Performance optimization testing
OPTIMIZE dbr_feature_test ZORDER BY (category, created_date);



path,metrics
s3://ag-lakehouse-momo/uc/demo_catalog/raw/__unitystorage/schemas/1c2396b7-8cca-4e5e-9dda-ac4c3292025a/tables/a4a721e4-7611-4f2d-9c49-bc1c3b003c62,"List(1, 9, List(907039, 907039, 907039.0, 1, 907039), List(18155, 114958, 104177.77777777778, 9, 937600), 0, List(minCubeSize(107374182400), List(0, 0), List(9, 937600), 0, List(9, 937600), 1, null), null, 0, 1, 9, 0, false, 0, 0, 1759283007559, 1759283009647, 8, 1, null, List(8, 1918), null, 4, 4, 612, 0, null)"


In [0]:
%sql
-- Cleanup
DROP TABLE dbr_feature_test;